# Model and SystemModel

Goal: derive the shallow water equations as a `Model` and freeze them into a `SystemModel`.

In [1]:
import numpy as np
import sympy as sp

from zoomy_core import coords as C
import zoomy_core.derivatives as d
from zoomy_core.model.derivation import (
    Model, PDETransformation, Simplify, ResolveIntegral,
    Consolidate, InvertMassMatrix, ChangeOfVariables, FoldConservative)
from zoomy_core.model.derivation.projection import Integrate
from zoomy_core.model.derivation.system_extract import HydrostaticPressure
from zoomy_core.model.operations import Multiply, ProductRule, KinematicBC
from zoomy_core.model.operations import Integrate as IntegrateZ
from zoomy_core.model.models.equations import Mass, Momentum, moment_scaling

t, x, z = C.t, C.x, C.z
zeta = sp.Symbol("zeta", real=True)

model = Model(coords=(t, x, z), parameters={"g": 9.81, "rho": 1.0})
g = model.parameters.g
h = sp.Function("h", positive=True)(t, x)
b = sp.Function("b", real=True)(t, x)

model.declare_state(h)
model.add_equation("bottom", d.t(b))
model.add_equation(Mass(model))
momentum = Momentum(model)
model.add_equation(momentum)
moment_scaling(model, momentum)
u, w, p = momentum.uvel[0], momentum.w, momentum.p
model.move_to_aux(b)
model.add_equation("kbc_top", KinematicBC(w=w, u=u, interface=b + h))
model.add_equation("kbc_bot", KinematicBC(w=w, u=u, interface=b))

model.apply({model.functions.tau_xz.expr: 0})

mz = model.momentum.z
mz.apply({d.t(w): 0, d.z(w * w): 0, d.x(u * w): 0})
mz.apply(IntegrateZ(z, z, b + h, method="analytical"))
mz.apply({p.subs(z, b + h): 0})
pressure = mz.solve_for(p)
model.momentum.x.apply(pressure)
mz.remove()
model.momentum.x.apply(Simplify())

model.apply(PDETransformation({z: (zeta, sp.Eq(z, b + h * zeta))}))

ubar = sp.Function(r"\bar{u}", real=True)(t, x)
utilde = model.functions.u.head
for eq in model._equations.values():
    eq.expr = eq.expr.replace(utilde, lambda *a: ubar)
for a in model._assumptions.values():
    a._as_relation = {k.replace(utilde, lambda *aa: ubar):
                      v.replace(utilde, lambda *aa: ubar)
                      for k, v in a._as_relation.items()}
model.redeclare_unknown(utilde, ubar, rename_key=True)

for e in (model.mass, model.momentum.x):
    e.apply(Multiply(h))
    e.apply(ProductRule(variables=[zeta]))
    e.apply(Integrate(zeta, bounds=(0, 1)))
    e.apply(ResolveIntegral())
    e.apply(model.kbc_bot)
    e.apply(model.kbc_top)
    e.apply({sp.Derivative(b, t): 0})
    e.apply(Simplify())

h_eq = model.mass.solve_for(sp.Derivative(h, t))
model.momentum.x.apply(h_eq)
model.momentum.x.apply(Consolidate())
model.apply(ChangeOfVariables(r"\bar{u}", "q_0", lambda q: q / h))
model.apply(InvertMassMatrix())
q0 = sp.Function("q_0", real=True)(t, x)
model.momentum.x.apply(FoldConservative([q0], h=h, b=b, x=x, gravity=g))

P_hydro = g * h**2 / 2
model.momentum.x.expr = model.momentum.x.expr.replace(
    lambda e: isinstance(e, sp.Derivative) and e.args[0] == P_hydro,
    lambda e: sp.Derivative(HydrostaticPressure(P_hydro), *e.args[1:],
                            evaluate=False))

model.describe()

**model** — 3 equations (1 vector), 30 ops — coords ('t', 'x', 'zeta')
**Parameters:** $g = 9.81$, $\rho = 1.0$, $e_{x} = 0.0$
**Q (declared):** `h`, `q_0`

**Equations:**
- **momentum** (vector):
  - **momentum_x:** $g \frac{\partial}{\partial x} b{\left(t,x \right)} h{\left(t,x \right)} - e_{x} g h{\left(t,x \right)} + \frac{\partial}{\partial x} \left(\frac{q_{0}^{2}{\left(t,x \right)}}{h{\left(t,x \right)}}\right) + \frac{\partial}{\partial x} \operatorname{HydrostaticPressure}{\left(\frac{g h^{2}{\left(t,x \right)}}{2} \right)} + \frac{\partial}{\partial t} q_{0}{\left(t,x \right)} = 0$
- **bottom:** $\frac{\partial}{\partial t} b{\left(t,x \right)} = 0$
- **mass:** $\frac{\partial}{\partial t} h{\left(t,x \right)} + \frac{\partial}{\partial x} q_{0}{\left(t,x \right)} = 0$

In [2]:
from zoomy_core.systemmodel import SystemModel

sm = SystemModel.from_model(model, Q=[b, h, q0])
sm.describe()

**SystemModel** — 3 equations, 1 spatial dimension

**Parameters:** $g = 9.81$, $\rho = 1.0$, $e_{x} = 0.0$

**System form:**

$$M(Q)\,\partial_t Q + \nabla\cdot\!\big(F(Q) + P(Q)\big) + \sum_{d} B_{d}(Q)\,\partial_{d} Q - \nabla\cdot\!\big(A(Q)\,\nabla Q\big) - S(Q) = 0$$

**State $Q$:**

$$
\left[\begin{matrix}b\\h\\q_{0}\end{matrix}\right]
$$

**Auxiliary $Q_{aux} = \varnothing$**

**Mass matrix $M$:**

$$
\left[\begin{matrix}1 & 0 & 0\\0 & 1 & 0\\0 & 0 & 1\end{matrix}\right]
$$

**Flux $F$:**

$$
\left[\begin{matrix}0\\q_{0}\\\frac{q_{0}^{2}}{h}\end{matrix}\right]
$$

**Hydrostatic pressure $P$:**

$$
\left[\begin{matrix}0\\0\\\frac{g h^{2}}{2}\end{matrix}\right]
$$

**Diffusive flux $A$ $= 0$**

**NCP $B$:**

$$
\left[\begin{matrix}0 & 0 & 0\\0 & 0 & 0\\g h & 0 & 0\end{matrix}\right]
$$

**Source $S$ (implicit):**

$$
\left[\begin{matrix}0\\0\\e_{x} g h\end{matrix}\right]
$$

**Operations:** from_model, declare_roles, declare_tags

In [3]:
from zoomy_core.model.analysis import ModelAnalyser

analyser = ModelAnalyser(sm)
eps = analyser.get_eps()
h0, u0 = sp.Symbol("h_0", positive=True), sp.Symbol("u_0", real=True)
fh, fq = analyser.create_functions_from_list(["h", "q"])
linear = analyser.linearize_system(
    sp.Matrix([0, h0 + eps * fh, h0 * u0 + eps * fq]), sp.zeros(0, 1))
analyser.delete_equations([i for i, e in enumerate(linear) if e is sp.true])
analyser.insert_plane_wave_ansatz([fh, fq])
analyser.remove_exponential()
omegas = analyser.solve_for_dispersion_relation()
for omega in omegas:
    display(sp.Eq(sp.Symbol("omega"), sp.simplify(omega)))

Eq(omega, k_x*(-sqrt(g)*sqrt(h_0) + u_0))

Eq(omega, k_x*(sqrt(g)*sqrt(h_0) + u_0))

In [4]:
values = {"k_x": 2.0, "g": 9.81, "h_0": 1.0, "u_0": 0.5}
got = sorted(float(w.subs({s: values[s.name] for s in w.free_symbols})) for w in omegas)
c = 2.0 * 9.81 ** 0.5
assert sm.n_state == 3 and np.allclose(got, [1.0 - c, 1.0 + c])